# 04 · Is our Switching observer the paper's model?

**Goal.** Confirm, step by step, that `observers/models/switching_observer.py`
implements the *same model* Laquitaine & Gardner (2018) define — not just
something that fits well and is loosely "switch-like".

This is a **different question** from "does the model work" (that's the
validation notebook, `03`). Here we do not fit anything and we barely touch
data. We take the paper's mathematical definition of the Switching observer,
break it into its defining pieces, and check our code reproduces each piece.

**How to read this notebook.** Each step has three parts:
1. **Paper** — what Laquitaine & Gardner say the model does (Methods, "Switching observer").
2. **Check** — a small runnable cell that interrogates our code for that property.
3. **Why this proves it** — what the result establishes about correspondence.

The paper's definition (Methods, STAR Methods "Switching observer") reduces to
**six defining properties**. If our code has all six, it is that model.


## Setup

In [ ]:
import sys, os, numpy as np
REPO = os.path.abspath(os.getcwd())
if "observers" not in os.listdir(REPO):
    REPO = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, REPO); os.chdir(REPO)
from observers.models.switching_observer import SwitchingObserver
obs = SwitchingObserver()   # paper's fitted-free defaults
print("loaded SwitchingObserver from", REPO)
print("parameters:", list(obs.__dataclass_fields__.keys()))

## The six defining properties

From the paper's Methods, the Switching observer is specified by these six
statements (paraphrased; the exact wording is in the PDF, `docs/…switching_observer.pdf`, Methods):

1. **Same ingredients as the Basic Bayesian observer** — a von Mises sensory
   likelihood (one concentration per coherence) and a von Mises prior at the
   experimental prior mean (one concentration per block width).
2. **It does NOT multiply prior × likelihood.** "Rather than forming a posterior
   by multiplying prior and likelihood, the observer switched between prior mean
   and evidence based on their relative strength."
3. **The switch probabilities are a reliability ratio** — `p_prior` and `p_e`
   set by the relative strengths (concentrations) of prior and evidence.
4. **The prior arm is a delta function at the prior mean** — "a delta function
   over percepts that peaks at the prior mean."
5. **The result is convolved with von Mises motor noise** (concentration `k_motor`).
6. **Lapses**: with probability `p_r` the estimate is random.

We check each in turn.


## Property 1 — same ingredients: von Mises likelihood (per coherence) + von Mises prior at the prior mean (per width)

**Paper.** The Switching observer inherits the Basic Bayesian observer's
sensory evidence distribution — a von Mises whose concentration rises with
coherence — and a learnt prior, a von Mises at the experimental prior mean
(225°) whose concentration encodes the block width.

**Check.** Our parameters should be: one `k_like` per coherence (increasing
with coherence), one `k_prior` per block width (increasing as width shrinks),
and a prior mean of 225.


In [ ]:
print("prior mean (deg):", obs.prior_mean, " <- experimental prior mean")
print("k_like (coherence -> sensory concentration):", obs.k_like)
print("k_prior (block width label -> prior concentration):", obs.k_prior)
print()
# ordering the paper requires
kl = obs.k_like
print("sensory reliability rises with coherence:", kl[0.06] < kl[0.12] < kl[0.24])
kp = obs.k_prior
print("prior reliability rises as width shrinks (80<40<20<10):",
      kp['80'] < kp['40'] < kp['20'] < kp['10'])
# implied prior SDs (deg) should roughly track the block labels
print("implied prior SDs (deg):", {k: round(v,1) for k,v in obs.prior_sd_degrees().items()})

**Why this proves it.** The switch model must be built from exactly the same
sensory + prior ingredients as the Basic Bayesian observer — that's what makes
"switch vs integrate" a fair, single-difference comparison. We have one sensory
concentration per coherence, one prior concentration per width, both ordered the
way reliability should go, and the prior sits at 225°. Same ingredients.


## Properties 2 & 4 — it switches (no multiplication); the two arms are the pure evidence read-out and a delta at the prior mean

**Paper.** The defining move: *instead of* multiplying prior and likelihood into
one posterior (that's the Basic Bayesian / integration observer), the Switching
observer **switches between the prior mean and the evidence**. So its two
"arms" are:
- the **evidence arm** — the sensory read-out with the prior switched OFF, and
- the **prior arm** — a **delta function at the prior mean** (not a broad prior).

**Check.** We interrogate the two arms directly. Set the switch weight all the
way to evidence, then all the way to prior, and look at the resulting estimate
distribution (motor noise and lapse off so we see the raw arms).


In [ ]:
# a clean, low-motor, no-lapse observer so we see the raw arms
raw = SwitchingObserver(p_random=0.0, k_motor=1e6, k_cardinal=0.0)
theta_true, coh, width = 135, 0.06, '40'   # stimulus far from prior (225)

# evidence arm: force k_prior -> 0 so the prior can't pull  (switch = all evidence)
ev = SwitchingObserver(p_random=0.0, k_motor=1e6,
        k_prior={'80':0.0,'40':0.0,'20':0.0,'10':0.0})
d_ev = np.asarray(ev.estimate_distribution(theta_true, coh, width)); d_ev/=d_ev.sum()

# prior arm: force k_like -> 0 so evidence carries no information (switch = all prior)
pr = SwitchingObserver(p_random=0.0, k_motor=1e6,
        k_like={0.06:0.0,0.12:0.0,0.24:0.0})
d_pr = np.asarray(pr.estimate_distribution(theta_true, coh, width)); d_pr/=d_pr.sum()

def peak(p): return int(p.argmax()+1)
print(f"stimulus = {theta_true} deg, prior mean = 225 deg")
print(f"evidence arm peaks at: {peak(d_ev)}  (should be ~{theta_true}, the stimulus)")
print(f"prior arm    peaks at: {peak(d_pr)}  (should be 225, the prior mean)")
# is the prior arm a DELTA (all mass on one direction)?
print(f"prior arm mass exactly at 225: {d_pr[224]:.3f}  (delta -> ~1.0)")
eff = np.exp(-(d_pr[d_pr>0]*np.log(d_pr[d_pr>0])).sum())
print(f"prior arm effective support: {eff:.2f} directions  (delta -> 1.0)")

**Why this proves it.** The two arms are exactly what the paper specifies: with
the prior off, the estimate follows the **stimulus** (pure evidence read-out);
with evidence off, the estimate collapses to a **delta at 225°**, the prior
mean. Crucially the prior arm is a *spike at the mean*, not a broad von Mises —
that is the "delta function over percepts that peaks at the prior mean" in the
paper, and it is what distinguishes switching from integration. Our code never
forms a single multiplied posterior; it reads out the two arms separately and
mixes them. That is Property 2 and Property 4.


## Property 3 — the switch probability is the reliability ratio `p_prior = k_prior / (k_prior + k_e)`

**Paper.** The observer chooses the prior arm with probability `p_prior` and the
evidence arm with probability `p_e`, set by their **relative strength**. In the
paper's parameterization this is the reliability (concentration) ratio.

**Check.** Call the model's own weight function and compare it to the closed-form
ratio, across a range of concentrations.


In [ ]:
# our internal weight routine vs the paper's closed form
print(" k_prior  k_e |  p_prior(code)  p_prior(ratio)   match")
for k_p, k_e in [(2.7, 1.0), (0.5, 8.0), (8.7, 3.0), (1.4, 1.4), (5.0, 0.0)]:
    w = raw._switching_weights(k_e, k_p)     # returns (w_prior, w_evidence, w_random) up to renorm
    w_prior_code = w[0] / (w[0] + w[1]) if (w[0]+w[1]) > 0 else float('nan')
    ratio = k_p / (k_p + k_e) if (k_p + k_e) > 0 else float('nan')
    ok = np.isclose(w_prior_code, ratio, atol=1e-9) or (np.isnan(ratio))
    print(f"  {k_p:4.1f}  {k_e:4.1f} |    {w_prior_code:6.4f}       {ratio:6.4f}      {ok}")

**Why this proves it.** The model's own switch weight equals `k_prior /
(k_prior + k_e)` exactly — the paper's reliability ratio — for every
combination. When the prior is more concentrated than the evidence the observer
leans prior; when the evidence is sharper it leans evidence. This is the
paper's Equation-6 switching law, not an approximation of it.


## Properties 5 & 6 — von Mises motor noise, and a lapse rate

**Paper.** The switched read-out is **convolved with von Mises motor noise**
(concentration `k_motor`), and **lapses** occur with probability `p_r` (a
fraction of trials estimated at random / uniform).

**Check.** Turn each knob and confirm it does what the paper says: raising
motor noise should *broaden* the estimate distribution; raising the lapse rate
should lift a *uniform floor* under it.


In [ ]:
base = SwitchingObserver(p_random=0.0, k_motor=40.0)
sharp = SwitchingObserver(p_random=0.0, k_motor=200.0)   # less motor noise -> sharper
d_base  = np.asarray(base.estimate_distribution(135,0.24,'10')); d_base/=d_base.sum()
d_sharp = np.asarray(sharp.estimate_distribution(135,0.24,'10')); d_sharp/=d_sharp.sum()
def spread(p): return float(np.exp(-(p[p>0]*np.log(p[p>0])).sum()))
print(f"motor noise: k_motor=40 spread={spread(d_base):.1f} dirs vs k_motor=200 spread={spread(d_sharp):.1f} dirs")
print(f"  -> more motor noise = broader distribution: {spread(d_base) > spread(d_sharp)}")

# lapse lifts a uniform floor
no_lapse = SwitchingObserver(p_random=0.0)
lapse    = SwitchingObserver(p_random=0.20)
d0 = np.asarray(no_lapse.estimate_distribution(135,0.24,'10')); d0/=d0.sum()
d1 = np.asarray(lapse.estimate_distribution(135,0.24,'10')); d1/=d1.sum()
# a far-from-everything direction should gain mass under lapses
probe = 45  # far from stimulus(135) and prior(225)
print(f"lapse: P(estimate near {probe} deg) no-lapse={d0[probe-1]:.5f}  lapse0.20={d1[probe-1]:.5f}")
print(f"  -> lapse raises the uniform floor: {d1[probe-1] > d0[probe-1]}")

**Why this proves it.** Motor noise acts as a von Mises convolution — more of
it broadens the predicted estimates — and the lapse rate mixes in a uniform
component that lifts probability everywhere, including directions far from both
the stimulus and the prior. Both knobs behave exactly as the paper's Equations
for motor noise and lapse require.


## Putting it together — the assembled estimate distribution

The paper's full estimate distribution for the Switching observer is:

```
p(estimate | stimulus) =
    V(estimate; 0, k_motor)  ⊛  [ (1 - p_r) · ( p_e · p(θ_e | θ_true)         # evidence arm
                                              + p_prior · δ(θ - μ_prior) )    # prior arm (delta)
                                 + p_r · uniform ]                            # lapse
```

Every symbol on the right is a property we just checked: the two arms
(Properties 2 & 4), the `p_prior`/`p_e` reliability split (Property 3), the
motor convolution `⊛ V(·;0,k_motor)` (Property 5), and the `p_r` lapse
(Property 6) — all built from the same von Mises ingredients (Property 1).


## The independent check — the verification suite

Everything above interrogates the model one property at a time. The repo's
verification suite does the same correspondence checks as automated assertions,
including two we didn't do by hand: that the evidence arm equals a *pure* von
Mises read-out to machine precision, and that the switch weights match the
reliability ratio exactly. Running it is the one-command version of this whole
notebook.


In [ ]:
import observers.verification.verify_switching as vs
ok = vs.run()
print("\nverification suite passed:", ok)

## Conclusion — does our code implement the paper's Switching observer?

| # | Paper property | Our code |
|---|---|---|
| 1 | von Mises likelihood (per coh) + von Mises prior at 225 (per width) | ✓ ordered concentrations, prior mean 225 |
| 2 | switches instead of multiplying prior×likelihood | ✓ two separate arms, no single posterior |
| 3 | `p_prior = k_prior/(k_prior+k_e)` reliability ratio | ✓ exact match to closed form |
| 4 | prior arm = delta at the prior mean | ✓ spike at 225 (effective support = 1) |
| 5 | von Mises motor-noise convolution | ✓ broadens with k_motor |
| 6 | lapse rate `p_r` (uniform floor) | ✓ lifts uniform component |

All six defining properties hold, and the verification suite passes. Our
`switching_observer.py` **is** Laquitaine & Gardner's Switching observer — not
merely a model that switches, but the specific one they define.

**What this notebook did NOT do** (on purpose): it did not fit the model to
data, and it did not compare it to competitors. Correspondence to the paper is a
statement about the *model definition*; whether it fits your subjects best is
the separate question that notebook `03` (validation) and the model-comparison
work address.
